# Notebook 04: Modelado No Supervisado y Deep Learning

Este notebook aplica clustering no supervisado (K-Means) y opcionalmente un autoencoder para explorar patrones en la producción de pozos no convencionales.

## 1. Importar librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
# Configuración de gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Cargar el dataset con features engineering

In [ ]:
# Cargar dataset con features engineering
df = pd.read_csv('data/processed/produccion_con_features.csv')
print('Dataset shape:', df.shape)
df.head()

## 3. Seleccionar variables relevantes para clustering

In [ ]:
# Seleccionar variables relevantes para clustering
# Excluir identificadores, fechas, y columnas que puedan causar fuga
feature_cols = [
    'antiguedad_meses',
    'prod_pet_mes_anterior',
    'prod_gas_mes_anterior',
    'declinacion_pet',
    'declinacion_gas',
    'ratio_gas_petroleo',
    'prod_pet_acumulada',
    'prod_gas_acumulada',
    'profundidad',
    'formacion_cod',
    'cuenca_cod'
]
X = df[feature_cols].copy()
print('Features seleccionadas:', X.shape[1])
print('Valores nulos por columna:')
print(X.isnull().sum())

## 4. Imputar valores nulos y escalar características

In [ ]:
# Imputar nulos restantes con 0 (o mediana) si los hubiera
X = X.fillna(0)
# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)
print('Datos escalados shape:', X_scaled.shape)

## 5. Método del codo para elegir número de clusters

In [ ]:
# Método del codo para elegir número de clusters
inertias = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure()
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del codo para selección de k')
plt.show()

## 6. Silhouette score para validar calidad de clusters

In [ ]:
# Silhouette score para validar calidad
silhouette_scores = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

plt.figure()
plt.plot(K_range, silhouette_scores, 'go-')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs número de clusters')
plt.show()

best_k = K_range[np.argmax(silhouette_scores)]
print(f'Mejor k según silhouette: {best_k}')

## 7. Aplicar K‑Means con el mejor k

In [ ]:
# Aplicar K-Means con el mejor k
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)
df['cluster'] = cluster_labels

print('Distribución de clusters:')
print(df['cluster'].value_counts().sort_index())

## 8. Visualizar clusters usando PCA para reducción a 2D

In [ ]:
# Visualizar clusters usando PCA para reducción a 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure()
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=cluster_labels, cmap='viridis', alpha=0.6)
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.title('Clusters visualizados con PCA')
plt.colorbar(scatter, label='Cluster')
plt.show()

print(f'Varianza explicada por las 2 primeras PCs: {pca.explained_variance_ratio_.sum():.2f}')

## 9. Parte opcional: Autoencoder para reducción de dimensionalidad (Deep Learning)

Si se dispone de TensorFlow/Keras, se puede entrenar un autoencoder simple para comprimir las características y luego aplicar clustering en el espacio latent.

In [ ]:
# Intentar importar TensorFlow; si falla, mostrar mensaje informativo
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print(f'TensorFlow version: {tf.__version__}')
    # Definir autoencoder
    input_dim = X_scaled.shape[1]
    encoding_dim = 8  # dimensión del espacio latent
    
    input_layer = keras.Input(shape=(input_dim,))
    encoder = layers.Dense(encoding_dim, activation='relu')(input_layer)
    encoder = layers.Dense(encoding_dim//2, activation='relu')(encoder)
    decoder = layers.Dense(encoding_dim//2, activation='relu')(encoder)
    decoder = layers.Dense(input_dim, activation='linear')(decoder)
    
    autoencoder = keras.Model(input_layer, decoder)
    encoder_model = keras.Model(input_layer, encoder)
    
    autoencoder.compile(optimizer='adam', loss='mse')
    
    # Entrenar
    history = autoencoder.fit(
        X_scaled.values, X_scaled.values,
        epochs=50,
        batch_size=256,
        shuffle=True,
        validation_split=0.1,
        verbose=0    )
    
    # Visualizar pérdida
    plt.figure()
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Val')
    plt.xlabel('Época')
    plt.ylabel('Pérdida MSE')
    plt.title('Pérdida del Autoencoder')
    plt.legend()
    plt.show()
    
    # Obtener representaciones latent
    X_latent = encoder_model.predict(X_scaled.values)
    print(f'Representación latent shape: {X_latent.shape}')
    
    # Opcional: aplicar K-Means en el espacio latent
    kmeans_latent = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    latent_labels = kmeans_latent.fit_predict(X_latent)
    df['cluster_latent'] = latent_labels
    print('Distribución de clusters en espacio latent:')
    print(df['cluster_latent'].value_counts().sort_index())
    
    # Visualizar latent con PCA (aunque ya es bajo dimensional)
    plt.figure()
    plt.scatter(X_latent[:,0], X_latent[:,1], c=latent_labels, cmap='plasma', alpha=0.6)
    plt.xlabel('Latent dim 1')
    plt.ylabel('Latent dim 2')
    plt.title('Clusters en espacio latent (Autoencoder)')
    plt.colorbar(label='Cluster latent')
    plt.show()
    
except ImportError:
    print('TensorFlow no está instalado. Para usar la sección de Autoencoder, instálelo con:')
    print('pip install tensorflow')

## 10. Guardar modelos y resultados

In [ ]:
# Guardar modelos y resultados
import joblib
joblib.dump(kmeans_final, 'models/kmeans_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
print('Modelo KMeans y scaler guardados en models/')
df.to_csv('data/processed/produccion_con_features_clustered.csv', index=False)
print('Dataset con cluster guardado en data/processed/')

## Conclusión

En este notebook hemos realizado un análisis de clustering no supervisado sobre el dataset de producción de pozos no convencionales. Tras escalar las características seleccionadas, utilizamos el método del codo y el silhouette score para determinar el número óptimo de clusters. Luego aplicamos K‑Means y visualizamos los resultados mediante PCA. Además, incluimos una sección opcional para entrenar un autoencoder que reduzca la dimensionalidad y permita clustering en el espacio latent, lo que puede capturar relaciones no lineales más complejas. Los modelos y el dataset con las etiquetas de cluster se guardan para su uso posterior en el proyecto. Este análisis ayuda a identificar grupos de pozos con comportamientos similares de producción, lo que puede informar estrategias de operación y mantenimiento.